# PEFT Dual-Repo Smoke Test (Colab)

Notebook này dùng để kiểm tra nhanh flow mới:
- Push adapter lên repo adapter
- Merge full precision rồi push repo merged
- Reload repo merged từ Hub và infer verify

> Chạy trên Colab GPU (T4/A10/A100).

In [ ]:
import os

REPO_URL = os.getenv("CAPSTONE_REPO_URL", "https://github.com/DDkaa/Capstone_backend.git")
REPO_DIR = "/content/Capstone_backend"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

In [ ]:
!nvidia-smi

In [ ]:
import os

BRANCH = os.getenv("CAPSTONE_BRANCH", "Dka")
!git checkout {BRANCH}

In [ ]:
!pip -q install -U pip
!pip -q install python-dotenv

In [ ]:
!pip -q install -r pipeline/services/peft_finetuning/requirements.txt

In [ ]:
import os
from getpass import getpass

hf_token = None

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.getenv("HF_TOKEN")

if not hf_token:
    hf_token = getpass("Enter HF_TOKEN: " ).strip()

os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)

In [ ]:
BASE_MODEL = "nvidia/AceMath-1.5B-Instruct"
DATASET_WORKSPACE = "quangne"
DATASET_REPO = "geometry"

# Đổi thành workspace/repo của bạn trên Hugging Face
ADAPTER_REPO_ID = "your-hf-username/text2diagram-acemath-1_5b-peft-adapter-smoke"
MERGED_REPO_ID = "your-hf-username/text2diagram-acemath-1_5b-merged-smoke"

VERIFY_INPUT_TEXT = "Cho tam giác ABC, M là trung điểm của BC. Trên tia đối của BA lấy điểm N sao cho BN = AB. Gọi I là giao điểm MN và AC. Chứng minh AI = 2IC"

print("Adapter repo:", ADAPTER_REPO_ID)
print("Merged repo:", MERGED_REPO_ID)

In [ ]:
import shlex
import subprocess

cmd = [
    "python", "-m", "pipeline.services.peft_finetuning.finetune",
    "--model_name", BASE_MODEL,
    "--dataset_huggingface_workspace", DATASET_WORKSPACE,
    "--dataset_huggingface_repo_name", DATASET_REPO,
    "--adapter_repo_id", ADAPTER_REPO_ID,
    "--merged_repo_id", MERGED_REPO_ID,
    "--verify_from_hub", "true",
    "--verify_input_text", VERIFY_INPUT_TEXT,
    "--is_dummy", "true",
    "--dummy_train_samples", "50",
    "--dummy_eval_samples", "10",
    "--num_train_epochs", "1",
    "--per_device_train_batch_size", "1",
    "--per_device_eval_batch_size", "1",
    "--gradient_accumulation_steps", "1",
    "--learning_rate", "2e-4",
]

print("Running command:\n", " ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)

In [ ]:
from huggingface_hub import list_repo_files

for repo_id in [ADAPTER_REPO_ID, MERGED_REPO_ID]:
    files = list_repo_files(repo_id)
    print(f"\nRepo: {repo_id}")
    print(f"Total files: {len(files)}")
    preview = files[:20]
    for f in preview:
        print(" -", f)

print("\nKiểm tra nhanh:")
print("- Adapter repo nên có adapter_config.json, adapter_model.safetensors")
print("- Merged repo nên có config.json, model.safetensors, tokenizer files")